In [1]:
from dgl import from_networkx
import dgl
import torch.nn as nn
import torch as th
import torch.nn.functional as F
import dgl.function as fn
import networkx as nx
import pandas as pd
import socket
import struct
import random
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import category_encoders as ce
import numpy as np
from sklearn.utils import class_weight

In [2]:
device = th.device('cuda' if th.cuda.is_available() else 'cpu')
graphs, _ = dgl.load_graphs('train_b.bin')
G = graphs[0]

In [3]:
G = G.to(device)
node_features = G.ndata['h']
edge_features = G.edata['h']
edge_label = G.edata['label']


In [4]:
print(node_features.shape[0])
print(edge_features.shape[0])

434633
711502


In [5]:
def compute_accuracy(pred, labels):
    return (pred.argmax(1) == labels).float().mean().item()

class SAGELayer(nn.Module):
    def __init__(self, ndim_in, edims, ndim_out ):
        super(SAGELayer, self).__init__()
        ### force to outut fix dimensions
        self.W_msg = nn.Linear(ndim_in + edims, ndim_out)
        ### apply weight
        self.W_apply = nn.Linear(ndim_in + ndim_out, ndim_out)
        

    def message_func(self, edges):
        return {'m': self.W_msg(th.cat([edges.data['h'], edges.src['h']], 1))}

    def forward(self, g_dgl, nfeats, efeats):
        with g_dgl.local_scope():
            g = g_dgl
            g.ndata['h'] = nfeats
            g.edata['h'] = efeats
            # Eq4
            g.update_all(self.message_func, fn.mean('m', 'h_neigh'))
            # Eq5          
            g.ndata['h'] = F.relu(self.W_apply(th.cat([g.ndata['h'], g.ndata['h_neigh']], 1)))
            return g.ndata['h']

class MLPPredictor(nn.Module):
    def __init__(self, in_features, edim, out_classes):
        super().__init__()
        self.W = nn.Linear(in_features * 2 + edim, out_classes)

    def apply_edges(self, edges):
        h_u = edges.src['h']
        h_v = edges.dst['h']
        h_e = edges.data['h']
        score = self.W(th.cat([h_u, h_e, h_v], 1))
        return {'score': score}

    def forward(self, graph, h, efeats):
        with graph.local_scope():
            graph.ndata['h'] = h
            graph.edata['h'] = efeats
            graph.apply_edges(self.apply_edges)
            return graph.edata['score']

class Model(nn.Module):
    def __init__(self,  ndim_in, ndim_out, edim):
        super().__init__()
        self.atten = nn.Parameter(th.randn(1, edim))
        self.cov1 = SAGELayer(ndim_in, edim, ndim_out)
        self.cov2 = SAGELayer(ndim_out, edim, ndim_out)
        self.pred = MLPPredictor(ndim_out, edim, 2)
        self.dropout = nn.Dropout(p=0.2)
        
    def forward(self, g, nfeats, efeats):
        efeats = efeats * self.atten
        nfeats = F.relu(self.cov1(g, nfeats, efeats))
        nfeats = self.dropout(nfeats)
        nfeats = F.relu(self.cov2(g, nfeats, efeats))
        nfeats = self.dropout(nfeats)
        return self.pred(g, nfeats, efeats)


In [6]:
criterion = nn.CrossEntropyLoss()
model = Model( G.ndata['h'].shape[1], 64, G.edata['h'].shape[1]).to(device)
opt = th.optim.Adam(model.parameters())

In [7]:
for epoch in range(10):
    pred = model( G, node_features, edge_features)
    loss = criterion(pred ,edge_label)
    opt.zero_grad()
    loss.backward()
    opt.step()
    if (epoch+1) % 1 == 0:
      print('Training acc:', compute_accuracy(pred, edge_label))

Training acc: 0.6103004813194275
Training acc: 0.6125646829605103
Training acc: 0.6305800676345825
Training acc: 0.6304255127906799
Training acc: 0.6955721974372864
Training acc: 0.6858898401260376
Training acc: 0.6663663387298584
Training acc: 0.6555891633033752
Training acc: 0.6598814725875854
Training acc: 0.6795440912246704


In [8]:
for epoch in range(50):
    pred = model( G, node_features, edge_features)
    loss = criterion(pred ,edge_label)
    opt.zero_grad()
    loss.backward()
    opt.step()
    if (epoch+1) % 10 == 0:
      print('Training acc:', compute_accuracy(pred, edge_label))

Training acc: 0.7436380982398987
Training acc: 0.7589381337165833
Training acc: 0.7948761582374573
Training acc: 0.8977444767951965
Training acc: 0.9268996715545654


In [9]:
for epoch in range(50):
    pred = model( G, node_features, edge_features)
    loss = criterion(pred ,edge_label)
    opt.zero_grad()
    loss.backward()
    opt.step()
    if (epoch+1) % 10 == 0:
      print('Training acc:', compute_accuracy(pred, edge_label))

Training acc: 0.9410050511360168
Training acc: 0.9465342164039612
Training acc: 0.9510654807090759
Training acc: 0.9457246661186218
Training acc: 0.9541519284248352


In [11]:
for epoch in range(20):
    pred = model( G, node_features, edge_features)
    loss = criterion(pred ,edge_label)
    opt.zero_grad()
    loss.backward()
    opt.step()
    if (epoch+1) % 10 == 0:
      print('Training acc:', compute_accuracy(pred, edge_label))

Training acc: 0.9642910361289978
Training acc: 0.9644217491149902


In [12]:
graphs, _ = dgl.load_graphs('test_b.bin')
G_test = graphs[0]

In [13]:
G_test = G_test.to(device)
node_features_test = G_test.ndata['h']
edge_features_test = G_test.edata['h']
edge_label_test = G_test.edata['label']


In [14]:
print(node_features_test.shape[0])
print(edge_features_test.shape[0])

193917
304930


In [15]:
#64-4100-0.06
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
out = model( G_test, node_features_test, edge_features_test)
out = F.softmax(out, dim=1)
_, pred = out.max(dim=1)
acc = accuracy_score(edge_label_test.cpu().numpy(), pred.cpu().numpy())
precision = precision_score(edge_label_test.cpu().numpy(), pred.cpu().numpy())
recall = recall_score(edge_label_test.cpu().numpy(), pred.cpu().numpy())
f1 = f1_score(edge_label_test.cpu().numpy(), pred.cpu().numpy())
print(f'Accuracy: {acc:.4f}, Precision: {precision:.4f}, Recall: {recall:.4f}, F1-score: {f1:.4f}')

Accuracy: 0.9654, Precision: 0.9701, Recall: 0.9761, F1-score: 0.9731
